<a href="https://colab.research.google.com/github/anhelus/alsia-workshop/blob/master/Notebooks/01_you_only_harvest_once.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/anhelus/alsia-workshop.git
!mv alsia-workshop/Notebooks/* .

# You Only Harvest Once

In questo notebook di accompagnamento, affronteremo i seguenti argomenti.

1. *Configurazione dell'ambiente di sviluppo*
2. *Utilizzo dei modelli di object detection*
3. *Utilizzo pratico dei risultati ottenuti*
4. *Oltre l'object detection: i modelli open world*

## 1. Configurazione dell'ambiente di sviluppo

Un programma in Python necessita spesso di installare una serie di *dipendenze*, anche dette *librerie*, che forniscono funzionalità aggiuntive rispetto a quelle "standard" fornite dalla libreria standard.

In questo notebook, la libreria che utilizzeremo è [Ultralytics](https://docs.ultralytics.com/). Questa libreria, open source e liberamente accessibile su GitHub, può essere installata mediante un package manager come `pip`, che è quello che utilizzeremo in questo caso specifico.

In [ ]:
!pip install ultralytics

Ultralytics ci mette a disposizione tutto il necessario per utilizzare dei modelli di object detection basati su YOLO (ma non solo). In tal senso, dovremo seguire un semplice flusso di lavoro; in particolare, dovremo:

1. Creare un modello utilizzando la classe `YOLO`.
2. Addestrare (preferibilmente) il modello creato al punto precedente sui nostri dati.
3. Usare il modello addestrato per effettuare la predizione.

Vediamo come fare.

## 2. Utilizzo dei modelli di object detection

### 2.1 Inizializzazione del modello

Come già detto, un modello di object detection in Ultralytics prevede la creazione di un oggetto di classe `YOLO` (o, nel caso si utilizzi un detector basato su transformer, di classe `RT-DETR`).

Per farlo, dovremo in primis importare la classe `YOLO` dal package `ultralytics`:

In [ ]:
from ultralytics import YOLO

A questo punto, dovremo creare un *oggetto* di classe `YOLO`, passandogli i pesi del modello che vogliamo inizializzare. Per farlo, potremo scegliere tra due opzioni:

* *usare un modello preaddestrato*, usando un file dei pesi con estensione `.pt`;
* *usare un modello non preaddestrato*, usando un file di configurazione con estensione `.yaml`.

La differenza tra le due scelte è decisiva. Un modello preaddestrato potrà individuare all'interno di un'immagine soltanto oggetti appartenenti ad una delle classi presenti all'interno del dataset su cui è stato addestrato; un modello non preaddestrato, invece, dovrà necessariamente subire un processo di *training* su un nuovo dataset per dare risultati coerenti.

In questo primo test, utilizzeremo un modello *preaddestrato* appartenente alla famiglia YOLOv11; in particolare, sceglieremo il modello più "leggero" della famiglia, ovvero quello con densità *nano*.

In [ ]:
model = YOLO('models/yolo11n-seg.pt')

### 2.2 Predizione zero-shot

A questo punto avremo creato un oggetto chiamato `model` che potremo utilizzare per effettuare predizioni sulle nostre immagini. Per farlo, dovremo utilizzare il metodo `predict` come segue:

In [ ]:
results = model.predict(source="data/lettuce")

Come risultato di questa operazione avremo ottenuto _results_, ovvero una lista di oggetti di tipo `Result`, uno per ogni immagine presente nella cartella indicata come `source`. I `Result` sono oggetti strutturati che permettono di leggere e visualizzare i risultati delle predizioni in vari formati e modalità.

### 2.3 Visualizzazione dei risultati

Il metodo `predict` mette a disposizione il parametro `show` per mostrare tutti i risultati ottenuti, ma per maggiore chiarezza possiamo creare una nostra visualizzazione personalizzata sfruttando il metodo `plot` degli oggetti `Result`. Implementiamo questa visualizzazione nella funzione `show_yolo_results`:

In [ ]:
# Definizione di show_yolo_results()

import matplotlib.pyplot as plt

def show_yolo_results(results):
    plt.figure(figsize=(10, 10))

    for idx, result in enumerate(results[:4]):
        annotated = result.plot(show=False)
        plt.subplot(2, 2, idx+1)
        plt.imshow(annotated[..., ::-1])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

Possiamo quindi passare ad applicare la nostra visualizzazione personalizzata sui results che abbiamo ottenuto precedentemente:

In [ ]:
show_yolo_results(results)

### Esercizio #1
Proviamo a rilevare dei pomodori sulle immagini in `data/tomato`!

In [ ]:
# Esercizio 1
results = model.predict(source="data/tomato")
show_yolo_results(results)

### 2.4 Predizione con modelli fine-tuned

A differenza dell'esempio precedente, in cui abbiamo provato ad rilevare le piante con un modello addestrato su dati generici (persone, auto, cani...), ora proviamo a vedere come i risultati cambiano con un modello addestrato su un dataset appropriato:

In [ ]:
model = YOLO('models/yolo11n-seg-lettuce.pt')
results = model.predict(source="data/lettuce")
show_yolo_results(results)

La differenza è molto grande! Abbiamo quindi dimostrato che usare modelli specializzati può aiutare in maniera importante per applicazioni particolarmente specifiche.

## 3. Utilizzo pratico dei risultati ottenuti

Come visto in precedenza, la libreria Ultralytics salva i risultati della predizione all'interno della variabile `results`. Questa variabile non è un semplice dato di tipo numerico: è una *variabile strutturata* che permette di accedere ai risultati in termini di object detection (o di segmentazione). Per esempio, potremmo pensare di conteggiare gli oggetti di un certo tipo all'interno di un'immagine:

In [ ]:
len(results[0].boxes)

In questo caso stiamo accedendo al primo elemento dei risultati, corrispondente alla prima immagine fornita. La numerazione inizia da 0, quindi ad esempio all'indice 3 corrisponde il quarto oggetto, all'indice 5 il sesto e così via. Con l'attributo `boxes` accediamo ai risultati delle rilevazioni per l'immagine, rappresentati da una lista di quintuple (classe, x_centro, y_centro, larghezza, altezza) degli oggetti trovati dal modello. La funzione Python `len` restituisce il numero di elementi nella lista.

### Esercizio #2
Proviamo a vedere quante lattughe ci sono nell'immagine `lettuce-6.jpg`!

In [ ]:
# Esercizio 2
results = model.predict(source="data/lettuce/lettuce-6.png")
len(results[0].boxes)

## 4. Oltre l'object detection: i modelli open world

Abbiamo visto come un modello standard di object detection può essere usato o considerando i pesi individuati dopo l'addestramento su grandi dataset, oppure riaddestrando il modello da zero su un certo dataset di riferimento.

Tuttavia, vedremo nelle prossime esercitazioni che organizzare un dataset solitamente richiede un grande impegno: potrebbe essere infatti necessario dover *etichettare* (ovvero, contrassegnare manualmente) tutti gli oggetti di interesse presenti all'interno dell'immagine, di modo da passare al modello i dati nel formato corretto. Il "costo" di questa operazione è però stato mitigato nel tempo grazie all'utilizzo dei cosiddetti modelli *open world*, che partono da una serie di *prompt* (ovvero istruzioni), che possono essere sia visive (indicando le zone di interesse nell'immagini) sia testuali (scrivendo ciò che si vuole cercare), per dare una prima "scrematura" dei dati presenti all'interno dell'immagine.

Un modello di questo tipo è, ad esempio, *YOLO World*, anch'esso disponibile all'interno della libreria Ultralytics, che usa dei prompt di tipo testuale per far individuare al modello esattamente gli oggetti appartenenti alle classi che desideriamo. Ad esempio, supponendo di voler individuare tutte le lattughe nell'immagine `lettuce.png`, potremo scrivere:

In [ ]:
open_model = YOLO("models/yolov8s-world.pt")
# Definiamo le classi di interesse
open_model.set_classes(["lettuce"])
# Usiamo il metodo predict come al solito
results = open_model.predict("data/lettuce/lettuce-2.png")
show_yolo_results(results)

Proviamo a vedere quante lattughe vengono individuate!

In [ ]:
len(results[0].boxes)

Il modello non è stato neanche in grado di individuare le lattughe in primo piano. Come mai?

### Few-shot prediction: prompt singolo

Quando si usano modelli che integrano prompt testuali è molto importante fare attenzione al livello di "descrittività" che tali modelli si aspettano nelle indicazioni degli oggetti da trovare.
Nel caso di YOLO World, il modello di linguaggio sottostante si chiama _CLIP_ (Contrastive Language-Image Pre-training), il quale è addestrato su prompt con un livello più elevato di descrittività del semplice termine "lettuce". Proviamo a modificare il nostro prompt:

In [ ]:
open_model = YOLO("models/yolov8s-world.pt")
open_model.set_classes(["a crop of lettuce"])  # Sostituiamo la classe con un prompt più descrittivo
results = open_model.predict("data/lettuce/lettuce-2.png")
show_yolo_results(results)

Bene! Ora il modello ha trovato una lattuga. Quella sulla destra, però, non viene ancora rilevata! Come possiamo fare?

### Few-shot prediction: prompt doppio

Innanzitutto proviamo ad abbassare la soglia di confidenza che il modello usa per filtrare le predizioni, in modo tale da essere più permissivi sugli oggetti rilevati.

Inoltre, possiamo provare ad espandere il nostro prompt introducendo una seconda indicazione, in modo tale da fornire al modello più informazioni su come una lattuga possa apparire nell'immagine. In questo caso gli diremo di cercare piante di lattughe (come prima) e, più in generale, "piccole piante verdi":

In [ ]:
open_model = YOLO("models/yolov8s-world.pt")
open_model.set_classes(["a crop of lettuce", "a small green plant"])  # Aggiungiamo un secondo prompt per cercare di catturare più piante
# Aggiungiamo il parametro 'conf' per abbassare la soglia di confidenza
results = open_model.predict("data/lettuce/lettuce-2.png", conf=0.1)
show_yolo_results(results)

### Few-shot prediction: prompt ensembling

Questo stratagemma di unire più prompt diversi riferiti alla stessa classe semantica si ispira ai cosiddetti modelli _ensembling_. Normalmente, l'ensembling consiste nel creare più versioni diverse dello stesso modello e nell'usarle per fare inferenza sullo stesso input, indipendentemente l'una dall'altra. L'output finale deriva quindi dalla selezione o dall'unione di questi risultati indipendenti, garantendo una maggiore robustezza di tale output.

Nei modelli open world applichiamo un procedimento simile quando andiamo a fornire più indicazioni, in quanto l'inferenza dell'object detector su ogni declinazione del prompt rappresenta formalmente una variante indipendente del modello; andando poi ad unire i risultati delle diverse rilevazioni, otteniamo un output più completo. Un esempio più strutturato di prompt ensembling è il seguente:

In [ ]:
open_model = YOLO("models/yolov8s-world.pt")
open_model.set_classes([
    "a lettuce plant",
    "a photo of lettuce",
    "a head of lettuce",
    "a lettuce seedling",
    "a young lettuce plant",
    "a small green plant",
    "lettuce growing in soil",
    "a photo of lettuce from above",
    "a rosette of green leaves",
    "a leafy green vegetable",
    "a crop of lettuce"
])
results = open_model.predict("data/lettuce/lettuce-2.png", conf=0.05)
show_yolo_results(results)

Bene, ora abbiamo delle rilevazioni complete e coerenti con i nostri obiettivi. È importante notare come i modelli open world consentano moltissime possibilità in base alle diverse combinazioni di addestramento, prompt, parametri, ecc.

L'approccio consigliato, quindi, rimane quello di sperimentare con varie configurazioni e capire quale sia la migliore per la propria applicazione pratica.